In [ ]:
# @title Criação do dataset - Execute primeiro antes de tudo
import pandas as pd
import random
import numpy as np

# Configurações para garantir que os dados sejam sempre iguais (reprodutibilidade)
random.seed(42)
np.random.seed(42)

# --- CONFIGURAÇÃO DOS DADOS SIMULADOS ---

# 1. Dados para Lei de Lotka (Autores)
# Poucos autores produzem muito, muitos produzem pouco
autores_prolificos = [f"Autor_Top_{i}" for i in range(1, 6)] # 5 autores 'estrelas'
autores_medios = [f"Autor_Medio_{i}" for i in range(1, 21)] # 20 autores médios
autores_ocasionais = [f"Autor_Unico_{i}" for i in range(1, 101)] # 100 autores de um artigo só

# Peso na escolha (Autores Top têm 20x mais chance de aparecer)
lista_autores_ponderada = (autores_prolificos * 20) + (autores_medios * 5) + (autores_ocasionais * 1)

# 2. Dados para Lei de Bradford (Revistas)
revistas_core = ["Journal of Data Science", "Scientometrics Today", "Information Processing & Management"]
revistas_zona2 = [f"Revista Tecnica {i}" for i in range(1, 8)]
revistas_zona3 = [f"Boletim Cientifico {i}" for i in range(1, 21)]

# Peso na escolha (Revistas Core têm 50x mais chance de aparecer)
lista_revistas_ponderada = (revistas_core * 50) + (revistas_zona2 * 15) + (revistas_zona3 * 2)

# 3. Dados para Lei de Zipf (Palavras no Título)
palavras_comuns = ["analysis", "study", "data", "of", "the", "and", "in", "system"]
palavras_tecnicas = ["algorithm", "network", "neural", "bibliometrics", "citation", "mining", "cloud", "ai"]
palavras_raras = ["heterogeneity", "stochastic", "disambiguation", "epistemology", "ontology"]

def gerar_titulo():
    p1 = random.choice(palavras_tecnicas + palavras_comuns)
    p2 = random.choice(palavras_comuns)
    p3 = random.choice(palavras_tecnicas)
    p4 = random.choice(palavras_raras + palavras_tecnicas)
    return f"{p1.capitalize()} {p2} {p3} {p4}"

# --- GERAÇÃO DO DATAFRAME E ARQUIVO ---

n_registros = 500
dados = {
    'ID_Artigo': range(1, n_registros + 1),
    'Titulo': [gerar_titulo() for _ in range(n_registros)],
    'Autor_Principal': [random.choice(lista_autores_ponderada) for _ in range(n_registros)],
    'Revista': [random.choice(lista_revistas_ponderada) for _ in range(n_registros)],
    'Ano': [random.randint(2018, 2024) for _ in range(n_registros)]
}

df_simulado = pd.DataFrame(dados)

# Salva o arquivo no disco
nome_arquivo = 'dados_bibliometricos_aula.csv'
df_simulado.to_csv(nome_arquivo, index=False)

print(f"Sucesso! O arquivo '{nome_arquivo}' foi criado com 500 registros (menu lateral esquerdo: Arquivos).")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 1. CARREGAR OS DADOS
# Certifique-se de que o arquivo 'dados_bibliometricos_aula.csv' está na mesma pasta
df = pd.read_csv('dados_bibliometricos_aula.csv')

print("--- Visão Geral dos Dados ---")
df.info()

In [ ]:
# ==============================================================================
# ANÁLISE 1: LEI DE BRADFORD (Revistas)
# ==============================================================================

import matplotlib.pyplot as plt

# Contagem de artigos por revista
bradford_data = df['Revista'].value_counts().reset_index()
bradford_data.columns = ['Revista', 'Artigos']

# Cálculo acumulado
bradford_data['Acumulado'] = bradford_data['Artigos'].cumsum()
bradford_data['Porcentagem'] = (bradford_data['Acumulado'] / bradford_data['Artigos'].sum()) * 100

# Identificando o Núcleo (Top 33%)
nucleo = bradford_data[bradford_data['Porcentagem'] <= 33]
print(f"Revistas do Núcleo (concentram 1/3 da produção):\n{nucleo['Revista'].tolist()}")

# Gráfico de Bradford (com salvamento)
plt.figure(figsize=(10, 5))
plt.plot(range(1, len(bradford_data) + 1), bradford_data['Porcentagem'], marker='o')
plt.axhline(y=33, color='r', linestyle='--', label='Núcleo (33%)')
plt.title('Lei de Bradford: Concentração de Revistas')
plt.xlabel('Ranking das Revistas')
plt.ylabel('% Acumulada de Artigos')
plt.legend()
plt.grid(True)

# Ajusta layout e salva ANTES de mostrar
plt.tight_layout()
plt.savefig("lei_de_bradford_revistas.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ==============================================================================
# ANÁLISE 2: LEI DE LOTKA (Autores)
# ==============================================================================

import matplotlib.pyplot as plt

print("\n--- 2. Análise de Lotka (Produtividade dos Autores) ---")

# Contagem de artigos por autor
lotka_data = df['Autor_Principal'].value_counts().reset_index()
lotka_data.columns = ['Autor', 'Qtd_Artigos']

# Distribuição de Frequência (Quantos autores publicaram X artigos?)
distribuicao_lotka = lotka_data['Qtd_Artigos'].value_counts().sort_index()

print("Distribuição de Produtividade (X Artigos -> Y Autores):")
print(distribuicao_lotka.head())

# Gráfico de Lotka (com salvamento)
plt.figure(figsize=(10, 5))
distribuicao_lotka.plot(kind='bar', color='skyblue')
plt.title('Lei de Lotka: Produtividade Científica')
plt.xlabel('Número de Artigos Publicados')
plt.ylabel('Quantidade de Autores')
plt.grid(axis='y')

# Ajusta layout e salva ANTES de mostrar
plt.tight_layout()
plt.savefig("lei_de_lotka_autores.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# ==============================================================================
# ANÁLISE 3: LEI DE ZIPF (Palavras nos Títulos)
# ==============================================================================
print("\n--- 3. Análise de Zipf (Mineração de Títulos) ---")

# Juntando todos os títulos em um único texto
todos_titulos = ' '.join(df['Titulo']).lower()
palavras = todos_titulos.split()

# Contagem de frequência
zipf_data = pd.Series(palavras).value_counts().reset_index()
zipf_data.columns = ['Palavra', 'Frequencia']
zipf_data['Ranking'] = zipf_data.index + 1

print("Top 10 Palavras mais usadas:")
print(zipf_data.head(10))

# Gráfico de Zipf
plt.figure(figsize=(10, 5))
# Mostrando apenas as top 20 palavras para não poluir
plt.bar(zipf_data['Palavra'].head(20), zipf_data['Frequencia'].head(20))
plt.title('Lei de Zipf: Frequência de Termos nos Títulos')
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
# ==============================================================================
# ANÁLISE 3: LEI DE ZIPF (Palavras nos Títulos)
# ==============================================================================

import pandas as pd
import matplotlib.pyplot as plt

print("\n--- 3. Análise de Zipf (COM LIMPEZA DE DADOS) ---")

# 1. Preparar o texto
todos_titulos = ' '.join(df['Titulo'].astype(str)).lower()
palavras = todos_titulos.split()

print(f"Total de palavras antes da limpeza: {len(palavras)}")

# 2. DEFINIR STOP WORDS (A lista de palavras proibidas)
# Como nossos dados simulados estão em inglês (ex: "Analysis", "Data"), usamos stop words em inglês.
# Se fossem dados reais em português, usaríamos: ['de', 'a', 'o', 'que', 'em', ...]
stop_words = [
    'of', 'the', 'and', 'in', 'to', 'for', 'with', 'on', 'at', 'by',
    'from', 'up', 'about', 'into', 'over', 'after'
]

# 3. FILTRAR (List Comprehension)
# Tradução: "Mantenha a palavra P se P NÃO ESTIVER na lista stop_words"
palavras_limpas = [p for p in palavras if p not in stop_words]

print(f"Total de palavras após limpeza: {len(palavras_limpas)}")

# 4. Contagem e Ranking (Igual ao anterior)
zipf_data = pd.Series(palavras_limpas).value_counts().reset_index()
zipf_data.columns = ['Palavra', 'Frequencia']
zipf_data['Ranking'] = zipf_data.index + 1

print("\nTop 10 Palavras RELEVANTES (Assuntos Reais):")
print(zipf_data.head(10))

# 5. Visualização Comparativa (com salvamento)
plt.figure(figsize=(12, 6))
plt.bar(zipf_data['Palavra'].head(15), zipf_data['Frequencia'].head(15), color='green')
plt.title('Lei de Zipf Pós-Limpeza: Os Reais Tópicos de Pesquisa')
plt.xlabel('Palavras-chave')
plt.ylabel('Frequência')
plt.xticks(rotation=45, ha='right')

# Ajusta layout e salva ANTES de mostrar
plt.tight_layout()
plt.savefig("lei_de_zipf_palavras_no_titulo.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
print("\n--- Análise Detalhada das Zonas de Bradford ---")

# A variável 'nucleo' já contém as revistas do Núcleo (aproximadamente 33%)
print(f"\nRevistas do Núcleo (Zona 1 - até 33% acumulado):\n{[row['Revista'] for index, row in nucleo.iterrows()]}")

# Identificar a Segunda Zona (33% < Porcentagem <= 66%)
segunda_zona = bradford_data[(bradford_data['Porcentagem'] > 33) & (bradford_data['Porcentagem'] <= 66)]
print(f"\nRevistas da Segunda Zona (entre 33% e 66% acumulado):\n{segunda_zona['Revista'].tolist()}")

# Identificar a Terceira Zona (Porcentagem > 66%)
terceira_zona = bradford_data[bradford_data['Porcentagem'] > 66]
print(f"\nRevistas da Terceira Zona (acima de 66% acumulado):\n{terceira_zona['Revista'].tolist()}")

print("\nIsso conclui a categorização das revistas nas três zonas da Lei de Bradford.")

In [ ]:
print("\n--- Quantificação da Concentração de Artigos por Zona ---")

# Quantificação do Núcleo (Zona 1)
num_revistas_nucleo = len(nucleo)
artigos_nucleo = nucleo['Artigos'].sum()
porcentagem_nucleo = nucleo['Porcentagem'].max() # O max do núcleo é o total acumulado do núcleo
print(f"\nNúcleo (Zona 1): {num_revistas_nucleo} revistas, {artigos_nucleo} artigos ({porcentagem_nucleo:.2f}% do total acumulado)")

# Quantificação da Segunda Zona
num_revistas_segunda_zona = len(segunda_zona)
artigos_segunda_zona = segunda_zona['Artigos'].sum()
# A porcentagem da segunda zona é o max da segunda zona menos o max do núcleo
porcentagem_acumulada_segunda_zona_total = segunda_zona['Porcentagem'].max()
porcentagem_contrib_segunda_zona = porcentagem_acumulada_segunda_zona_total - porcentagem_nucleo
print(f"Segunda Zona: {num_revistas_segunda_zona} revistas, {artigos_segunda_zona} artigos ({porcentagem_contrib_segunda_zona:.2f}% de contribuição, acumulando até {porcentagem_acumulada_segunda_zona_total:.2f}%)")

# Quantificação da Terceira Zona
num_revistas_terceira_zona = len(terceira_zona)
artigos_terceira_zona = terceira_zona['Artigos'].sum()
# A porcentagem da terceira zona é o max da terceira zona menos o max da segunda zona
porcentagem_acumulada_terceira_zona_total = terceira_zona['Porcentagem'].max()
porcentagem_contrib_terceira_zona = porcentagem_acumulada_terceira_zona_total - porcentagem_acumulada_segunda_zona_total
print(f"Terceira Zona: {num_revistas_terceira_zona} revistas, {artigos_terceira_zona} artigos ({porcentagem_contrib_terceira_zona:.2f}% de contribuição, acumulando até {porcentagem_acumulada_terceira_zona_total:.2f}%)")